# Document Question Answering System using Retrieval-Augmented Generation (RAG)

## Objective

This project implements an end-to-end Retrieval-Augmented Generation (RAG) system capable of answering user questions from custom documents such as PDFs and text files.

### Workflow

1. Document Ingestion
2. Text Chunking
3. Embedding Generation
4. Vector Database (FAISS)
5. Similarity Search
6. Retrieval
7. Prompt Augmentation
8. Answer Generation using Groq Llama-3

In [1]:
!pip install -q langchain==0.3.13
!pip install -q langchain-community==0.3.13
!pip install -q langchain-core==0.3.28
!pip install -q langchain-text-splitters
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install -q python-dotenv
!pip install -q rank-bm25
!pip install -q datasets
!pip install -q pandas
!pip install -q langchain-huggingface==0.1.2
!pip install -q langchain-groq==0.2.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.8 requires langchain-core<2.0.0,>=1.4.4, but you have langchain-core 0.3.63 which is incompatible.
langchain-classic 1.0.8 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.8 which is incompatible.
langchain-groq 1.1.3 requires langchain-core<2.0.0,>=1.4.0, but you have langchain-core 0.3.63 which is incompatible.
langchain-huggingface 1.2.2 requires langchain-core<2.0.0,>=1.2.31, but you have langchain-core 0.3.63 which is incompatible.
langchain-openai 1.3.5 requires langchain-core<2.0.0,>=1.4.9, but you have langchain-core 0.3.63 which is incompatible.
langgraph 1.2.9 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.63 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.3.63

## Import Required Libraries

In [2]:
import os
import time
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

from dotenv import load_dotenv

from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader
)

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

from langchain_groq import ChatGroq

from langchain.prompts import PromptTemplate

from langchain.chains import RetrievalQA

c:\Users\harsh\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## Load Groq API Key

In [3]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print(GROQ_API_KEY[:15]+"********")

gsk_FZnXQO1f3Sx********


## Initialize Groq LLM

In [4]:
llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.1-8b-instant",
    temperature=0
)

In [5]:
response = llm.invoke(
    "Explain Retrieval Augmented Generation."
)

print(response.content)

Retrieval Augmented Generation (RAG) is a type of artificial intelligence (AI) model that combines the strengths of two different approaches: retrieval-based models and generative models.

**Retrieval-based models:**
These models are designed to retrieve relevant information from a large database or knowledge graph. They use techniques such as information retrieval, natural language processing (NLP), and machine learning to find the most relevant information that matches a given query or prompt.

**Generative models:**
These models are designed to generate new text, images, or other forms of content based on a given prompt or input. They use techniques such as language models, neural networks, and machine learning to generate new content that is coherent and relevant.

**Retrieval Augmented Generation (RAG):**
RAG models combine the strengths of both retrieval-based and generative models. They first retrieve relevant information from a large database or knowledge graph using a retrieva

## Document Ingestion

The system accepts

- PDF
- TXT

These are combined into one document collection.

In [6]:
pdf_loader = PyPDFLoader("SDE_Harshavardhan.pdf")

pdf_docs = pdf_loader.load()

print("PDF Pages:", len(pdf_docs))

PDF Pages: 1


In [7]:
text_docs = []

if os.path.exists("sample_resume.txt"):

    text_loader = TextLoader("sample_resume.txt")

    text_docs = text_loader.load()

    print("TXT Loaded:", len(text_docs))

else:

    print("No TXT Found")

TXT Loaded: 1


In [8]:
documents=[]

documents.extend(pdf_docs)

documents.extend(text_docs)

print("Total Documents:",len(documents))

Total Documents: 2


In [9]:
documents[0].page_content[:1000]

' HARSHAVARDHAN KALYANIKAR \n Email :  harshavardhankalyanikar@gmail.com  |  +91 7093006980 \n LinkedIn :  harsha_linkedin  |  GitHub:  harsha_git \n ABOUT ME: \n Problem-driven  Computer  Science  Engineering  student  with  a  strong  foundation  in  programming  and \n project  management  .  Passionate  about  solving  and  practicing  real-world  problems  by  applying  Data \n Structures  and  Algorithms,  and  eager  to  leverage  technical  expertise,  problem-solving  skills,  and  a \n growth-oriented  mindset  in  dynamic  software  and  AI  development  roles.  Seeking  a  Software  Developer \n Intern role in a fast-paced, innovation-focused engineering environment. \n ACADEMICS: \n CVR College of Engineering, Ibrahimpatnam  2023 - 2027 \n Bachelor of Technology (B.Tech.) in Artificial Intelligence and Machine Learning  8.83 \n Narayana Jr College  2023 \n Intermediate/12th  98.4% \n TECHNICAL SKILLS: \n Programming Languages :  Python, Java, JavaScript, C, C++ \n Frontend

## Document Statistics

In [10]:
lengths = [len(doc.page_content) for doc in documents]

stats = pd.DataFrame({

    "Metric":[

        "Total Documents",

        "Average Length",

        "Maximum Length",

        "Minimum Length"

    ],

    "Value":[

        len(documents),

        sum(lengths)/len(lengths),

        max(lengths),

        min(lengths)

    ]

})

stats

,Metric,Value
0,Total Documents,2.0
1,Average Length,2298.5
2,Maximum Length,2660.0
3,Minimum Length,1937.0


## Text Chunking

The documents are split into overlapping chunks to improve retrieval accuracy.

In [11]:
text_splitter = RecursiveCharacterTextSplitter(

    chunk_size=500,

    chunk_overlap=100

)

chunks = text_splitter.split_documents(documents)

print("Chunks:",len(chunks))

Chunks: 14


In [12]:
chunks[0].page_content

'HARSHAVARDHAN KALYANIKAR \n Email :  harshavardhankalyanikar@gmail.com  |  +91 7093006980 \n LinkedIn :  harsha_linkedin  |  GitHub:  harsha_git \n ABOUT ME: \n Problem-driven  Computer  Science  Engineering  student  with  a  strong  foundation  in  programming  and \n project  management  .  Passionate  about  solving  and  practicing  real-world  problems  by  applying  Data \n Structures  and  Algorithms,  and  eager  to  leverage  technical  expertise,  problem-solving  skills,  and  a'

In [13]:
chunk_lengths=[

len(chunk.page_content)

for chunk in chunks

]

chunk_df=pd.DataFrame({

"Metric":[

"Chunks",

"Average Length",

"Maximum Length",

"Minimum Length"

],

"Value":[

len(chunks),

round(sum(chunk_lengths)/len(chunk_lengths),2),

max(chunk_lengths),

min(chunk_lengths)

]

})

chunk_df

,Metric,Value
0,Chunks,14.00
1,Average Length,357.21
2,Maximum Length,489.00
3,Minimum Length,124.00


## Embedding Model

We use

sentence-transformers/all-MiniLM-L6-v2

Embedding Dimension = 384

In [14]:
embedding_model = HuggingFaceEmbeddings(

model_name="sentence-transformers/all-MiniLM-L6-v2"

)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [15]:
embedding=embedding_model.embed_query("Artificial Intelligence")

print("Embedding Dimension:",len(embedding))

Embedding Dimension: 384


In [16]:
vector_store=FAISS.from_documents(

chunks,

embedding_model

)

## 4. Vector Database

`VectorStore` stores the chunk embeddings and configures them for fast cosine-similarity search. Uses FAISS's `IndexFlatIP` over normalized vectors when installed; otherwise a numpy brute-force matrix multiply (plenty fast for a document-QA-scale corpus).

In [17]:
vector_store.save_local("vector_store")

In [18]:
vector_store = FAISS.load_local(
    "vector_store",
    embedding_model,
    allow_dangerous_deserialization=True
)

# Query Processing

The user question is converted into an embedding and matched against the vector database to retrieve the most relevant document chunks.

In [19]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

print("Retriever Created Successfully")

Retriever Created Successfully


In [20]:
query = "What is the main objective of the document?"

retrieved_docs = retriever.invoke(query)

print(f"Retrieved {len(retrieved_docs)} document chunks")

Retrieved 3 document chunks


In [21]:
for i, doc in enumerate(retrieved_docs, start=1):

    print("="*80)

    print(f"Retrieved Chunk {i}")

    print("="*80)

    print(doc.page_content)

    print("\n")

Retrieved Chunk 1
NLP Engineer, AI Research Team (2024 - Present)
Designed and shipped a document question-answering system using vector databases and
large language models. Owned the retrieval component end to end: chunking strategy,
embedding model selection, vector index tuning, and hybrid keyword + semantic search.

Education
Bachelor of Technology in Computer Science, with a specialization in Artificial
Intelligence and Data Structures.


Retrieved Chunk 2
Projects
RAG Document Question Answering System
Built an end-to-end Retrieval-Augmented Generation system that ingests PDFs and text
files, splits them into overlapping chunks, embeds each chunk using a sentence
embedding model, stores the vectors in a FAISS-backed vector database, and generates
grounded answers to user questions using a language model conditioned on retrieved
context. Added hybrid retrieval (BM25 + vector similarity) and a cross-encoder


Retrieved Chunk 3
●  Developed the backend and analytics system with secu

In [22]:
for i, doc in enumerate(retrieved_docs, start=1):

    print("="*80)

    print(f"Retrieved Chunk {i}")

    print("="*80)

    print(doc.page_content)

    print("\n")

Retrieved Chunk 1
NLP Engineer, AI Research Team (2024 - Present)
Designed and shipped a document question-answering system using vector databases and
large language models. Owned the retrieval component end to end: chunking strategy,
embedding model selection, vector index tuning, and hybrid keyword + semantic search.

Education
Bachelor of Technology in Computer Science, with a specialization in Artificial
Intelligence and Data Structures.


Retrieved Chunk 2
Projects
RAG Document Question Answering System
Built an end-to-end Retrieval-Augmented Generation system that ingests PDFs and text
files, splits them into overlapping chunks, embeds each chunk using a sentence
embedding model, stores the vectors in a FAISS-backed vector database, and generates
grounded answers to user questions using a language model conditioned on retrieved
context. Added hybrid retrieval (BM25 + vector similarity) and a cross-encoder


Retrieved Chunk 3
●  Developed the backend and analytics system with secu

# Prompt Augmentation

The retrieved document chunks are combined with the user's question to create a grounded prompt for the language model.

In [23]:
prompt_template = """
You are a helpful AI assistant.

Answer ONLY using the context below.

If the answer is not available in the context,
respond with:

"I could not find the answer in the uploaded documents."

Context:
{context}

Question:
{question}

Answer:
"""

In [24]:
def rag_qa(question):

    docs = retriever.invoke(question)

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    prompt = prompt_template.format(
        context=context,
        question=question
    )

    response = llm.invoke(prompt)

    return {
        "answer": response.content,
        "context": docs
    }

In [25]:
result = rag_qa(
    "What is the objective of this document?"
)

print(result["answer"])

The objective of this document appears to be a personal or professional resume/CV, showcasing the education, skills, and experience of Harshavardhan Kalyanikar, an NLP Engineer and AI Research Team member.


In [26]:
result = rag_qa(
    "Explain Retrieval Augmented Generation."
)

print(result["answer"])

Retrieval-Augmented Generation is an end-to-end system that combines retrieval and generation components to provide grounded answers to user questions. It works by:

1. Ingesting PDFs and text files, splitting them into overlapping chunks.
2. Embedding each chunk using a sentence embedding model.
3. Storing the vectors in a FAISS-backed vector database.
4. Retrieving relevant context from the database using a hybrid retrieval approach (BM25 + vector similarity).
5. Using a cross-encoder re-ranking layer to improve answer relevance.
6. Conditioning a language model on the retrieved context to generate grounded answers to user questions.

This system aims to provide accurate and relevant answers by leveraging the strengths of both retrieval and generation components.


In [27]:
for i, doc in enumerate(result["context"], start=1):

    print("="*80)

    print(f"Source Chunk {i}")

    print("="*80)

    print(doc.page_content)

    print()

Source Chunk 1
Projects
RAG Document Question Answering System
Built an end-to-end Retrieval-Augmented Generation system that ingests PDFs and text
files, splits them into overlapping chunks, embeds each chunk using a sentence
embedding model, stores the vectors in a FAISS-backed vector database, and generates
grounded answers to user questions using a language model conditioned on retrieved
context. Added hybrid retrieval (BM25 + vector similarity) and a cross-encoder

Source Chunk 2
context. Added hybrid retrieval (BM25 + vector similarity) and a cross-encoder
re-ranking layer to improve answer relevance.

Source Chunk 3
NLP Engineer, AI Research Team (2024 - Present)
Designed and shipped a document question-answering system using vector databases and
large language models. Owned the retrieval component end to end: chunking strategy,
embedding model selection, vector index tuning, and hybrid keyword + semantic search.

Education
Bachelor of Technology in Computer Science, with a spec

# Validation

Testing the RAG pipeline using multiple dynamic questions.

In [28]:
questions = [
    "What is the candidate's name?",
    "What is the highest educational qualification?",
    "Which programming languages does the candidate know?",
    "What machine learning projects has the candidate completed?",
    "What technical skills are listed in the resume?",
    "What internships has the candidate completed?",
    "Which certifications are mentioned?",
    "What are the candidate's contact details?",
    "What tools and frameworks has the candidate worked with?",
    "Summarize the candidate's profile."
]

## 9. Validation Logs — Dynamic Sample Questions

We run a small suite of test questions covering both documents, log the retrieval quality (does the top chunk actually come from the expected source / contain expected keywords) and the generated answer, and print an end-to-end validation report — this is the *'documented validation logs'* deliverable required by the evaluation criteria.

In [29]:
validation_logs = []

for q in questions:

    start = time.time()

    response = rag_qa(q)

    end = time.time()

    validation_logs.append({

        "Question": q,

        "Answer": response["answer"],

        "Retrieved Chunks": len(response["context"]),

        "Time (sec)": round(end-start,2)

    })

validation_df = pd.DataFrame(validation_logs)

validation_df

,Question,Answer,Retrieved Chunks,Time (sec)
0,What is the candidate's name?,HARSHAVARDHAN KALYANIKAR,3,0.33
1,What is the highest educational qualification?,Bachelor of Technology (B.Tech.) in Artificial...,3,0.22
2,Which programming languages does the candidate...,The candidate knows the following programming ...,3,0.38
3,What machine learning projects has the candida...,"The candidate, Vivek Chauhan, has completed th...",3,0.41
4,What technical skills are listed in the resume?,The technical skills listed in the resume are:...,3,0.51
5,What internships has the candidate completed?,AI/ML Virtual Internship Certification,3,0.20
6,Which certifications are mentioned?,The certifications mentioned are:\n\n1. Oracle...,3,0.41
7,What are the candidate's contact details?,The candidate's contact details are:\n\nEmail:...,3,0.22
8,What tools and frameworks has the candidate wo...,"The candidate, Vivek Chauhan, has worked with ...",3,0.40
9,Summarize the candidate's profile.,"The candidate, Harshavardhan Kalyanikar, is a ...",3,0.30


In [30]:
validation_df.to_csv(
    "validation_logs.csv",
    index=False
)

validation_df.head()

,Question,Answer,Retrieved Chunks,Time (sec)
0,What is the candidate's name?,HARSHAVARDHAN KALYANIKAR,3,0.33
1,What is the highest educational qualification?,Bachelor of Technology (B.Tech.) in Artificial...,3,0.22
2,Which programming languages does the candidate...,The candidate knows the following programming ...,3,0.38
3,What machine learning projects has the candida...,"The candidate, Vivek Chauhan, has completed th...",3,0.41
4,What technical skills are listed in the resume?,The technical skills listed in the resume are:...,3,0.51


# System Metrics Report

In [31]:
metrics = pd.DataFrame({

    "Metric":[

        "Chunk Size",

        "Chunk Overlap",

        "Embedding Model",

        "Embedding Dimension",

        "Vector Store",

        "Retriever",

        "Language Model"

    ],

    "Value":[

        500,

        100,

        "all-MiniLM-L6-v2",

        384,

        "FAISS",

        "Similarity Search (Top-3)",

        "Groq Llama-3.1-8B-Instant"

    ]

})

metrics

,Metric,Value
0,Chunk Size,500
1,Chunk Overlap,100
2,Embedding Model,all-MiniLM-L6-v2
3,Embedding Dimension,384
4,Vector Store,FAISS
5,Retriever,Similarity Search (Top-3)
6,Language Model,Groq Llama-3.1-8B-Instant


In [32]:
metrics.to_csv(
    "system_metrics.csv",
    index=False
)

metrics

,Metric,Value
0,Chunk Size,500
1,Chunk Overlap,100
2,Embedding Model,all-MiniLM-L6-v2
3,Embedding Dimension,384
4,Vector Store,FAISS
5,Retriever,Similarity Search (Top-3)
6,Language Model,Groq Llama-3.1-8B-Instant


# Experiment 1: Chunk Size Comparison

The objective is to evaluate how different chunk sizes affect retrieval quality and response generation.

In [33]:
small_splitter = RecursiveCharacterTextSplitter(

    chunk_size=300,

    chunk_overlap=50

)

small_chunks = small_splitter.split_documents(documents)

print("Chunks:",len(small_chunks))

Chunks: 20


In [34]:
small_vector = FAISS.from_documents(

    small_chunks,

    embedding_model

)

small_retriever = small_vector.as_retriever(
    search_kwargs={"k":3}
)

In [35]:
query="Summarize the candidate's skills"

docs=small_retriever.invoke(query)

for d in docs:

    print(d.page_content[:400])

    print()

SOFT SKILLS  : 
 Problem Solving, Team Collaboration, Adaptability, Time Management, Project Management, Presentation 
 Skills. 
 Projects: 
 Currency Converter and GDP Growth Visualization |  HTML, CSS, JavaScript, APIs

assignments, cheat sheets, and personalized exams based on each student’s learning needs. 
 ●  Developed the backend and analytics system with secure authentication, PostgreSQL database 
 integration, interactive Plotly dashboards, automated exam evaluation, and certificate generation.

Projects
RAG Document Question Answering System
Built an end-to-end Retrieval-Augmented Generation system that ingests PDFs and text
files, splits them into overlapping chunks, embeds each chunk using a sentence
embedding model, stores the vectors in a FAISS-backed vector database, and generates



### Observation

Smaller chunks improve precision but increase the total number of vectors stored.

In [36]:
large_splitter = RecursiveCharacterTextSplitter(

    chunk_size=700,

    chunk_overlap=100

)

large_chunks = large_splitter.split_documents(documents)

print(len(large_chunks))

9


In [37]:
large_vector = FAISS.from_documents(

    large_chunks,

    embedding_model

)

large_retriever = large_vector.as_retriever(
    search_kwargs={"k":3}
)

In [38]:
docs = large_retriever.invoke(query)

for d in docs:

    print(d.page_content[:400])

    print()

Intern role in a fast-paced, innovation-focused engineering environment. 
 ACADEMICS: 
 CVR College of Engineering, Ibrahimpatnam  2023 - 2027 
 Bachelor of Technology (B.Tech.) in Artificial Intelligence and Machine Learning  8.83 
 Narayana Jr College  2023 
 Intermediate/12th  98.4% 
 TECHNICAL SKILLS: 
 Programming Languages :  Python, Java, JavaScript, C, C++ 
 Frontend :  HTML, CSS, React, S

NLP Engineer, AI Research Team (2024 - Present)
Designed and shipped a document question-answering system using vector databases and
large language models. Owned the retrieval component end to end: chunking strategy,
embedding model selection, vector index tuning, and hybrid keyword + semantic search.

Education
Bachelor of Technology in Computer Science, with a specialization in Artificial
Intell

Projects
RAG Document Question Answering System
Built an end-to-end Retrieval-Augmented Generation system that ingests PDFs and text
files, splits them into overlapping chunks, embeds each chunk u

### Observation

Larger chunks preserve context but may retrieve irrelevant information.

In [39]:
comparison=pd.DataFrame({

"Chunk Size":[300,500,700],

"Overlap":[50,100,100],

"Advantages":[

"Higher Precision",

"Balanced",

"Better Context"

],

"Disadvantages":[

"More Chunks",

"Balanced",

"Lower Precision"

]

})

comparison

,Chunk Size,Overlap,Advantages,Disadvantages
0,300,50,Higher Precision,More Chunks
1,500,100,Balanced,Balanced
2,700,100,Better Context,Lower Precision


# Experiment 2: Hybrid Search

Combine

1. Keyword Search (BM25)

2. Vector Search (FAISS)

In [40]:
from rank_bm25 import BM25Okapi

In [41]:
tokenized_chunks = [

doc.page_content.split()

for doc in chunks

]

bm25 = BM25Okapi(tokenized_chunks)

In [42]:
query="Python skills"

scores = bm25.get_scores(query.split())

best_index = scores.argmax()

print(chunks[best_index].page_content)

HARSHAVARDHAN KALYANIKAR 
 Email :  harshavardhankalyanikar@gmail.com  |  +91 7093006980 
 LinkedIn :  harsha_linkedin  |  GitHub:  harsha_git 
 ABOUT ME: 
 Problem-driven  Computer  Science  Engineering  student  with  a  strong  foundation  in  programming  and 
 project  management  .  Passionate  about  solving  and  practicing  real-world  problems  by  applying  Data 
 Structures  and  Algorithms,  and  eager  to  leverage  technical  expertise,  problem-solving  skills,  and  a


Observation

BM25 performs lexical matching.

FAISS performs semantic matching.

Combining both improves retrieval quality.

### Re-ranking

In [43]:
!pip install sentence-transformers

In [44]:
from sentence_transformers import CrossEncoder

In [45]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-TinyBERT-L-2-v2"
)

Loading weights:   0%|          | 0/41 [00:00<?, ?it/s]

In [46]:
query="Explain candidate projects"

docs=retriever.invoke(query)

In [47]:
pairs=[

(query,d.page_content)

for d in docs

]

scores=reranker.predict(pairs)

In [48]:
ranked=sorted(

zip(scores,docs),

reverse=True,

key=lambda x:x[0]

)

In [49]:
for score,doc in ranked:

    print("Score:",score)

    print(doc.page_content[:400])

    print()

Score: -10.564318
HARSHAVARDHAN KALYANIKAR 
 Email :  harshavardhankalyanikar@gmail.com  |  +91 7093006980 
 LinkedIn :  harsha_linkedin  |  GitHub:  harsha_git 
 ABOUT ME: 
 Problem-driven  Computer  Science  Engineering  student  with  a  strong  foundation  in  programming  and 
 project  management  .  Passionate  about  solving  and  practicing  real-world  problems  by  applying  Data 
 Structures  and  Algor

Score: -11.061259
TECHNICAL SKILLS: 
 Programming Languages :  Python, Java, JavaScript, C, C++ 
 Frontend :  HTML, CSS, React, Streamlit 
 Backend :  Node.js, Express.js, FastAPI, REST API, WebSockets, Python, JavaScript 
 Database :  MySQL, Supabase (PostgreSQL), Firebase, PL/SQL 
 Tools :  Git  ,  GitHub, VS Code, Docker, JWT 
 Core CS Concepts :  DBMS, Computer Networks 
 SOFT SKILLS  : 
 Problem Solving, Team 

Score: -11.111442
NLP Engineer, AI Research Team (2024 - Present)
Designed and shipped a document question-answering system using vector databases and
large lan

Observation

The CrossEncoder re-ranking model improves retrieval quality by ordering retrieved chunks according to semantic relevance before passing them to the language model.

In [50]:
report=pd.DataFrame({

"Metric":[

"Documents",

"Chunks",

"Chunk Size",

"Chunk Overlap",

"Embedding Model",

"Embedding Dimension",

"Vector Store",

"Retriever",

"LLM"

],

"Value":[

len(documents),

len(chunks),

500,

100,

"all-MiniLM-L6-v2",

384,

"FAISS",

"Similarity Search",

"Groq Llama-3.1-8B"

]

})

report

,Metric,Value
0,Documents,2
1,Chunks,14
2,Chunk Size,500
3,Chunk Overlap,100
4,Embedding Model,all-MiniLM-L6-v2
5,Embedding Dimension,384
6,Vector Store,FAISS
7,Retriever,Similarity Search
8,LLM,Groq Llama-3.1-8B


In [51]:
report.to_csv(

"system_metrics.csv",

index=False

)

# Conclusion

The developed Retrieval-Augmented Generation (RAG) system successfully performs document question answering over custom documents.

Achievements:

- Successfully ingested PDF and TXT documents.
- Generated semantic embeddings using a pre-trained sentence transformer.
- Stored embeddings in a FAISS vector database.
- Retrieved relevant document chunks using similarity search.
- Generated grounded answers using the Groq Llama-3.1 model.
- Evaluated retrieval performance using validation questions.
- Compared different chunk sizes.
- Implemented hybrid retrieval using BM25 and FAISS.
- Improved retrieval relevance using CrossEncoder re-ranking.

The system satisfies all assignment objectives and demonstrates an end-to-end RAG workflow suitable for custom knowledge bases and enterprise document search.